# HRA Portal — RUI & EUI Product Usage Analysis

**Course:** E538/E438 — Information Visualization, Indiana University  
**Role:** UI & Product Insights — Registration User Interface (RUI) + Exploration User Interface (EUI)  

---

## Overview
This notebook provides a deep-dive analysis of **how researchers actually use the RUI and EUI** interfaces of the Human Reference Atlas Portal. Using the cleaned CSV exports from DuckDB pipeline, we analyze:

- **RUI**: Usage trends over time, which organs are registered most, key workflow signals
- **EUI**: Exploration behavior, which features are adopted vs. ignored, user growth trends
- **Comparative**: RUI vs EUI engagement patterns and trajectory comparison

**Data Source:** Amazon CloudFront Logs (Oct 2023 – Jan 2026), processed via DuckDB on Parquet  
**Total Records in Dataset:** ~6.8M events across 28 months

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# ── Global Style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'font.family': 'DejaVu Sans',
})

# ── Colour palette ────────────────────────────────────────────────────────────
COLORS = {
    'RUI': '#1f77b4',   # strong blue
    'EUI': '#ff7f0e',   # warm orange
    'CDE': '#2ca02c',   # green
    'OTHER': '#cccccc', # grey
}

print('Libraries loaded successfully ✓')

In [ ]:
# ── Load the four exported CSVs ──────────────────────────────────────────────
# Update paths if running locally; these match the Colab export paths
monthly   = pd.read_csv('monthly_usage.csv')
feature   = pd.read_csv('feature_frequency.csv')
traffic   = pd.read_csv('traffic_summary.csv')
geo       = pd.read_csv('geography_summary.csv')

# ── Parse dates ──────────────────────────────────────────────────────────────
monthly['event_month'] = pd.to_datetime(monthly['event_month'])

print('Datasets loaded:')
print(f'  monthly_usage   : {monthly.shape}  — {monthly.event_month.min().strftime("%b %Y")} → {monthly.event_month.max().strftime("%b %Y")}')
print(f'  feature_frequency: {feature.shape}')
print(f'  traffic_summary : {traffic.shape}')
print(f'  geography_summary: {geo.shape}')

In [ ]:
# ── Convenience filters ──────────────────────────────────────────────────────
rui_m = monthly[monthly['interface_guess'] == 'RUI'].sort_values('event_month').reset_index(drop=True)
eui_m = monthly[monthly['interface_guess'] == 'EUI'].sort_values('event_month').reset_index(drop=True)
cde_m = monthly[monthly['interface_guess'] == 'CDE'].sort_values('event_month').reset_index(drop=True)

rui_f = feature[feature['interface_guess'] == 'RUI'].copy()
eui_f = feature[feature['interface_guess'] == 'EUI'].copy()

print('Filtered subsets created ✓')
print(f'  RUI months: {len(rui_m)}  |  EUI months: {len(eui_m)}  |  CDE months: {len(cde_m)}')
print(f'  RUI feature paths: {len(rui_f):,}  |  EUI feature paths: {len(eui_f):,}')

---
## 2. RUI Analysis — Registration User Interface

The RUI allows biomedical researchers to **spatially register tissue samples** onto 3D reference organs. Key signals in the CloudFront logs that indicate RUI activity are:
- `/api/v1/collisions` — real-time collision check when placing a tissue block on an organ model
- `/api/v1/extraction-site` — loading extraction site landmarks per organ
- `/hra-api/v1/get-spatial-placement` — retrieving spatial coordinates for a registered block
- `/api/v1/tissue-blocks` — fetching registered tissue blocks from the database
- `/landmark/*-extraction-sites/*` — per-organ landmark files

### 2.1 RUI Monthly Usage Trend (Oct 2023 – Jan 2026)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Event Count ────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(rui_m['event_month'], rui_m['event_count'],
         color=COLORS['RUI'], linewidth=2.5, marker='o', markersize=4, label='Events')

# Annotate the spike at Nov 2024
peak_idx = rui_m['event_count'].idxmax()
peak_row = rui_m.loc[peak_idx]
ax1.annotate(f"Peak: {int(peak_row['event_count']):,}\n{peak_row['event_month'].strftime('%b %Y')}",
             xy=(peak_row['event_month'], peak_row['event_count']),
             xytext=(peak_row['event_month'] - pd.DateOffset(months=5), peak_row['event_count'] * 0.85),
             arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5),
             fontsize=10, color='#333333',
             bbox=dict(boxstyle='round,pad=0.3', fc='#fff3cd', ec='#ffaa00', alpha=0.9))

# Annotate Sep 2025 secondary peak
sep25 = rui_m[rui_m['event_month'] == '2025-09-01']
if not sep25.empty:
    ax1.annotate(f"{int(sep25.iloc[0]['event_count']):,}\nSep 2025",
                 xy=(sep25.iloc[0]['event_month'], sep25.iloc[0]['event_count']),
                 xytext=(sep25.iloc[0]['event_month'] - pd.DateOffset(months=6), sep25.iloc[0]['event_count'] * 0.75),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=1.2),
                 fontsize=9, color='#555')

ax1.set_title('RUI — Monthly Event Count\n(Oct 2023 – Jan 2026)', pad=12)
ax1.set_xlabel('Month')
ax1.set_ylabel('Event Count')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.tick_params(axis='x', rotation=40)
ax1.fill_between(rui_m['event_month'], rui_m['event_count'], alpha=0.08, color=COLORS['RUI'])

# ── Right: Unique Users ──────────────────────────────────────────────────────
ax2 = axes[1]
ax2.bar(rui_m['event_month'], rui_m['unique_users'],
        width=20, color=COLORS['RUI'], alpha=0.75, label='Unique Users')
ax2.plot(rui_m['event_month'], rui_m['unique_users'],
         color=COLORS['RUI'], linewidth=1.5, linestyle='--', alpha=0.6)

ax2.set_title('RUI — Monthly Unique Users\n(Growing Researcher Base)', pad=12)
ax2.set_xlabel('Month')
ax2.set_ylabel('Unique Users')
ax2.tick_params(axis='x', rotation=40)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.suptitle('Fig 1. RUI Usage Trends — Event Volume & User Growth', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_rui_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKey Metrics:')
print(f'  Total RUI events (full period): {rui_m["event_count"].sum():,}')
print(f'  Peak month: {peak_row["event_month"].strftime("%b %Y")} — {int(peak_row["event_count"]):,} events')
print(f'  Unique users in 2024: {rui_m[rui_m.event_month.dt.year==2024]["unique_users"].sum():,}')
print(f'  Unique users in 2025: {rui_m[rui_m.event_month.dt.year==2025]["unique_users"].sum():,}')

**Insight:** RUI shows strong but episodic usage. The massive spike in **November 2024** (~189K events) strongly correlates with a HuBMAP Consortium event or a new dataset/version release that drove concentrated tissue registration activity. A secondary peak in **September 2025** (~52K events) shows the platform is actively growing. Unique users are on a clear upward trajectory from ~300/month (2023) to ~1,100–1,400/month (late 2025).

### 2.2 RUI Feature Usage — What Researchers Actually Do

Because CloudFront logs record HTTP requests rather than UI click events, we proxy user behavior through endpoint calls. For the RUI, four core endpoint families map clearly to user actions.

In [ ]:
# ── Categorise RUI endpoint families ────────────────────────────────────────
def categorize_rui(path):
    p = path.lower()
    if 'collision' in p:         return 'Tissue Collision Check'
    if 'extraction-site' in p:   return 'Extraction Site Load'
    if 'spatial-placement' in p: return 'Spatial Placement'
    if 'tissue-block' in p:      return 'Tissue Block Query'
    return 'Other RUI'

rui_f['category'] = rui_f['feature_proxy'].apply(categorize_rui)
rui_cat = rui_f.groupby('category').agg(
    total_events=('event_count', 'sum'),
    unique_users=('unique_users', 'sum')
).sort_values('total_events', ascending=True)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Horizontal bar — events
ax1 = axes[0]
colors_bar = ['#6baed6' if c != 'Tissue Collision Check' else '#08519c' for c in rui_cat.index]
bars = ax1.barh(rui_cat.index, rui_cat['total_events'], color=colors_bar, height=0.55, edgecolor='white')
for bar in bars:
    w = bar.get_width()
    ax1.text(w + 5000, bar.get_y() + bar.get_height()/2,
             f'{int(w):,}', va='center', fontsize=10, color='#333')
ax1.set_title('RUI Feature Categories — Total Events', pad=10)
ax1.set_xlabel('Total Event Count (all time)')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
ax1.set_xlim(0, rui_cat['total_events'].max() * 1.25)

# Right: Donut showing proportion
ax2 = axes[1]
donut_colors = ['#08519c', '#6baed6', '#bdd7e7', '#eff3ff', '#d0d0d0']
wedges, texts, autotexts = ax2.pie(
    rui_cat['total_events'][::-1],
    labels=rui_cat.index[::-1],
    autopct='%1.1f%%',
    startangle=90,
    colors=donut_colors,
    wedgeprops=dict(width=0.45, edgecolor='white'),
    pctdistance=0.75
)
for text in texts: text.set_fontsize(9)
for at in autotexts: at.set_fontsize(8)
ax2.set_title('RUI Feature Share\n(proportion of all RUI events)', pad=10)
ax2.text(0, 0, 'RUI', ha='center', va='center', fontsize=13, fontweight='bold', color='#333')

fig.suptitle('Fig 2. RUI Feature Usage Breakdown — What Researchers Do in the RUI', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_rui_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('RUI Feature Totals:')
print(rui_cat[['total_events','unique_users']].sort_values('total_events', ascending=False).to_string())

**Insight:** The **Tissue Collision Check** endpoint dominates RUI activity (53% of all RUI calls), which makes biological sense — every time a user moves or resizes a tissue block in the 3D viewer, the browser fires a collision API call in real time. This is a proxy for active engagement with the 3D placement tool. **Extraction Site Load** (27%) represents users selecting which organ to register against, and **Spatial Placement** (11%) captures completed registrations. The high Collision-to-Placement ratio (~4.7:1) suggests users spend significant time iterating on placement before saving.

### 2.3 RUI — Which Organs Are Registered Most?

Landmark endpoint paths encode the organ name directly in the URL (e.g., `/landmark/heart-female-extraction-sites/v1.0`), giving us a clear signal of which organs researchers are actively working with.

In [ ]:
# ── Extract organ names from landmark paths ──────────────────────────────────
landmark_paths = rui_f[rui_f['feature_proxy'].str.contains('landmark', case=False, na=False)].copy()

def extract_organ(path):
    m = re.search(r'/landmark/([a-z0-9-]+?)-(?:male|female|both)-extraction', path)
    if m:
        organ = m.group(1).replace('-', ' ').title()
        return organ
    return None

landmark_paths['organ'] = landmark_paths['feature_proxy'].apply(extract_organ)
organ_counts = landmark_paths.dropna(subset=['organ']).groupby('organ')['event_count'].sum().sort_values(ascending=False)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = ['#08519c' if i == 0 else '#6baed6' for i in range(len(organ_counts))]
bars = ax.bar(organ_counts.index, organ_counts.values, color=bar_colors, edgecolor='white', width=0.6)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 30,
            f'{int(h):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Fig 3. RUI — Which Organs Are Researchers Registering Most?\n(Landmark extraction-site endpoint calls as usage proxy)', pad=12)
ax.set_xlabel('Organ')
ax.set_ylabel('Total Landmark Endpoint Calls')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Add annotation
ax.annotate('Large Intestine & Spleen\nlead organ registrations',
            xy=(organ_counts.index[0], organ_counts.iloc[0]),
            xytext=(2.5, organ_counts.iloc[0] * 0.85),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5),
            fontsize=10, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', fc='#fff3cd', ec='#ffaa00', alpha=0.9))

plt.tight_layout()
plt.savefig('fig3_rui_organs.png', dpi=150, bbox_inches='tight')
plt.show()

print('Organ Usage Rankings:')
print(organ_counts.to_string())

**Insight:** The **Large Intestine** (female variant) leads organ registrations, followed by **Spleen** and **Kidney**. This pattern reflects active HuBMAP research priorities — gastrointestinal and lymphoid organ datasets are heavily studied. All eight organ variants show relatively balanced usage, suggesting the RUI is used across a range of tissue types rather than being dominated by a single research focus. The small variation between male/female variants confirms researchers are using the correct anatomically-matched models.

---
## 3. EUI Analysis — Exploration User Interface

The EUI enables researchers to **explore anatomical structures, cell types, and biomarkers** in a 3D reference organ viewer. Key endpoint signals:
- `/api/v1/scene` — 3D organ scene loading (core engagement action)
- `/api/v1/cell-type-term-occurences` — cell type filter usage
- `/api/v1/ontology-term-occurences` — anatomical structure ontology filter
- `/api/v1/biomarker-term-occurences` — biomarker filter usage
- `/api/v1/aggregate-results` — running aggregate queries across datasets
- `/api/v1/reference-organs` — loading the reference organ catalogue

### 3.1 EUI Monthly Usage Trend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Event count trend ──────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(eui_m['event_month'], eui_m['event_count'],
         color=COLORS['EUI'], linewidth=2.5, marker='s', markersize=4)
ax1.fill_between(eui_m['event_month'], eui_m['event_count'], alpha=0.1, color=COLORS['EUI'])

# Annotate Jan 2024 spike
jan24 = eui_m[eui_m['event_month'] == '2024-01-01']
if not jan24.empty:
    ax1.annotate(f"Jan 2024\n{int(jan24.iloc[0]['event_count']):,} events",
                 xy=(jan24.iloc[0]['event_month'], jan24.iloc[0]['event_count']),
                 xytext=(jan24.iloc[0]['event_month'] + pd.DateOffset(months=4), jan24.iloc[0]['event_count'] * 0.85),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=1.3),
                 fontsize=9, color='#555')

ax1.set_title('EUI — Monthly Event Count\n(Steady with Growing User Base)', pad=12)
ax1.set_xlabel('Month')
ax1.set_ylabel('Event Count')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.tick_params(axis='x', rotation=40)

# ── Right: Unique users — key story for EUI ──────────────────────────────────
ax2 = axes[1]
ax2.bar(eui_m['event_month'], eui_m['unique_users'],
        width=20, color=COLORS['EUI'], alpha=0.8)

# Draw trend line
from numpy.polynomial import polynomial as P
x_num = np.arange(len(eui_m))
coefs = np.polyfit(x_num, eui_m['unique_users'], deg=1)
trend = np.polyval(coefs, x_num)
ax2.plot(eui_m['event_month'], trend, color='red', linewidth=1.5, linestyle='--', label='Trend')
ax2.legend(fontsize=10)

ax2.set_title('EUI — Monthly Unique Users\n(Strong upward trend 2024→2025)', pad=12)
ax2.set_xlabel('Month')
ax2.set_ylabel('Unique Users')
ax2.tick_params(axis='x', rotation=40)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.suptitle('Fig 4. EUI Usage Trends — Event Volume & User Growth', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_eui_trends.png', dpi=150, bbox_inches='tight')
plt.show()

eui_2024_users = eui_m[eui_m.event_month.dt.year==2024]['unique_users'].sum()
eui_2025_users = eui_m[eui_m.event_month.dt.year==2025]['unique_users'].sum()
print(f'EUI unique users 2024: {eui_2024_users:,}')
print(f'EUI unique users 2025: {eui_2025_users:,}  (+{(eui_2025_users-eui_2024_users)/eui_2024_users*100:.1f}%)')
print(f'Peak user month: {eui_m.loc[eui_m.unique_users.idxmax(), "event_month"].strftime("%b %Y")} — {eui_m.unique_users.max():,} users')

**Insight:** The EUI tells a particularly compelling story on the **unique users** chart. While raw event counts remain fairly stable (10K–35K/month), unique users are on a **clear upward trajectory** — growing from ~450 users/month in 2024 to ~2,900 users/month by December 2025. This means the EUI is attracting more distinct researchers over time, even while session depth per user may be decreasing. This is a healthy growth signal: the EUI is becoming more widely adopted across the HuBMAP community.

### 3.2 EUI Feature Adoption — Which Tools Are Used vs. Ignored?

In [ ]:
# ── Categorise EUI endpoints into functional groups ──────────────────────────
def categorize_eui(path):
    p = path.lower()
    if 'aggregate-results' in p:     return 'Aggregate Results Query'
    if 'cell-type-tree' in p:        return 'Cell Type Tree (UI)'
    if 'cell-type-term' in p:        return 'Cell Type Filter'
    if 'ontology-tree' in p:         return 'Ontology Tree (UI)'
    if 'ontology-term' in p:         return 'Anatomical Structure Filter'
    if 'biomarker-tree' in p:        return 'Biomarker Tree (UI)'
    if 'biomarker-term' in p:        return 'Biomarker Filter'
    if 'reference-organ-scene' in p: return 'Reference Organ 3D Scene'
    if 'scene' in p:                 return 'Main 3D Scene'
    if 'reference-organ' in p:       return 'Reference Organ Catalogue'
    if 'technology-names' in p:      return 'Technology Filter'
    if 'provider-names' in p:        return 'Provider/Lab Filter'
    return 'Other EUI'

eui_f['category'] = eui_f['feature_proxy'].apply(categorize_eui)
eui_cat = eui_f.groupby('category').agg(
    total_events=('event_count', 'sum'),
    unique_users=('unique_users', 'sum')
).sort_values('total_events', ascending=True)

# Exclude 'Other EUI' from the chart for clarity
eui_cat_plot = eui_cat[eui_cat.index != 'Other EUI']

# ── Build color gradient ─────────────────────────────────────────────────────
n = len(eui_cat_plot)
bar_colors = [plt.cm.Blues(0.35 + 0.5 * (i / n)) for i in range(n)]

fig, ax = plt.subplots(figsize=(13, 7))

bars = ax.barh(eui_cat_plot.index, eui_cat_plot['total_events'],
               color=bar_colors, height=0.6, edgecolor='white')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 500, bar.get_y() + bar.get_height()/2,
            f'{int(w):,}', va='center', fontsize=9.5, color='#333')

ax.set_title('Fig 5. EUI Feature Adoption — Total Events per Functional Group\n'
             '(Higher bars = more heavily used; low bars = potential discoverability gap)',
             pad=14, fontsize=13)
ax.set_xlabel('Total Event Count (all time)', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
ax.set_xlim(0, eui_cat_plot['total_events'].max() * 1.2)

# Draw a divider between high and low adoption
median_idx = n // 2
ax.axhline(y=median_idx - 0.5, color='#e05c5c', linewidth=1.2, linestyle=':', alpha=0.7)
ax.text(eui_cat_plot['total_events'].max() * 1.05, median_idx - 0.4,
        'Relative\ndivide', fontsize=8, color='#e05c5c', va='bottom')

plt.tight_layout()
plt.savefig('fig5_eui_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('EUI Feature Categories:')
print(eui_cat[['total_events','unique_users']].sort_values('total_events', ascending=False).to_string())

**Insight:**
- **Cell Type Filter** and **Anatomical Structure Filter** top EUI usage — researchers primarily use the EUI to query cell distributions and structure annotations, which is the platform's core scientific value proposition.
- **Main 3D Scene** calls (29K) confirm the 3D organ viewer is actively used but not dominant — users enter it and then primarily use the filter panels.
- **Provider/Lab Filter** and **Technology Filter** are used roughly equally (~16K each), suggesting researchers do filter by data provenance but less so than by cell biology.
- **Tree UI components** (ontology tree, cell type tree, biomarker tree) all show substantial usage, confirming the hierarchical navigation panels are an important part of the EUI workflow.

### 3.3 EUI Spatial Search — How Often Do Researchers Use It?

One of the sponsor's explicit research questions: **How often is spatial search used in the EUI?**  
The `/api/v1/get-spatial-placement` and related spatial endpoints serve as the proxy.

In [ ]:
# ── Isolate spatial placement / spatial search signals ───────────────────────
spatial_paths = feature[
    feature['feature_proxy'].str.contains('spatial-placement|spatial_placement', case=False, na=False)
].copy()

# Also include do-search which is a KG-level search used by EUI
kg_search = feature[
    feature['feature_proxy'].str.contains('do-search', case=False, na=False) &
    (feature['interface_guess'].isin(['EUI', 'OTHER']))
].copy()

print('=== Spatial Placement Endpoint Calls ===')
print(spatial_paths[['interface_guess','feature_proxy','event_count','unique_users']]
      .sort_values('event_count', ascending=False).head(10).to_string())

total_spatial = spatial_paths['event_count'].sum()
total_rui_events = rui_f['event_count'].sum()
total_eui_events = eui_f['event_count'].sum()

print(f'\nTotal spatial-placement calls: {total_spatial:,}')
print(f'  As % of all RUI events: {total_spatial/total_rui_events*100:.1f}%')
print(f'\nKG search calls (proxy for EUI spatial search): {kg_search["event_count"].sum():,}')

In [ ]:
# ── Visualise: Spatial search in context of all RUI feature calls ────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Stacked context
ax1 = axes[0]
rui_feature_summary = {
    'Tissue Collision\n(placement interaction)': 436343,
    'Extraction Site\n(organ selection)': 223866,
    'Spatial Placement\n(GET spatial coords)': 93130,
    'Tissue Block\n(dataset queries)': 64904,
    'Other RUI': 537
}
colors_bar2 = ['#9ecae1','#6baed6','#2171b5','#084594','#cccccc']
keys = list(rui_feature_summary.keys())
vals = list(rui_feature_summary.values())
bars = ax1.barh(keys, vals, color=colors_bar2, height=0.55, edgecolor='white')
for bar in bars:
    w = bar.get_width()
    ax1.text(w + 3000, bar.get_y() + bar.get_height()/2,
             f'{int(w):,}', va='center', fontsize=10)
ax1.set_title('RUI Workflow Funnel\n(Collision → Placement represents the registration pipeline)', pad=10)
ax1.set_xlabel('Total Event Count')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
ax1.set_xlim(0, max(vals) * 1.25)

# Right: Spatial placement as proportion — donut
ax2 = axes[1]
spatial_events = 93130
non_spatial_rui = total_rui_events - spatial_events
ax2.pie(
    [spatial_events, non_spatial_rui],
    labels=['Spatial Placement\n(11.3%)', 'Other RUI Calls\n(88.7%)'],
    colors=['#2171b5', '#c6dbef'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.45, edgecolor='white'),
    pctdistance=0.75
)
ax2.set_title(f'Spatial Placement calls\nas share of all RUI traffic\n({spatial_events:,} of {total_rui_events:,} total)', pad=10)
ax2.text(0, 0, f'93K\nevents', ha='center', va='center', fontsize=11, fontweight='bold', color='#2171b5')

fig.suptitle('Fig 6. RUI Spatial Search/Placement — Usage Context & Frequency', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_spatial_search.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Spatial placement calls represent **11.3% of all RUI traffic** (~93K events total). This is a meaningful signal: spatial placement is the *output* of a successful tissue registration session. The funnel structure — Collision (436K) → Extraction Site (224K) → Spatial Placement (93K) — shows roughly a 4.7:1 and 2.4:1 drop-off at each stage. This is the expected shape of an iterative workflow where users explore multiple placement options before committing. The spatial search feature is being used consistently, not abandoned.

---
## 4. RUI vs EUI — Comparative Analysis

### 4.1 Side-by-Side Monthly Trends (All Three Interfaces)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# ── Top: Event counts ────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(rui_m['event_month'], rui_m['event_count'],
         color=COLORS['RUI'], linewidth=2.5, marker='o', markersize=3.5, label='RUI')
ax1.plot(eui_m['event_month'], eui_m['event_count'],
         color=COLORS['EUI'], linewidth=2.5, marker='s', markersize=3.5, label='EUI')
ax1.plot(cde_m['event_month'], cde_m['event_count'],
         color=COLORS['CDE'], linewidth=2, marker='^', markersize=3.5, label='CDE', linestyle='--')

ax1.fill_between(rui_m['event_month'], rui_m['event_count'], alpha=0.07, color=COLORS['RUI'])
ax1.fill_between(eui_m['event_month'], eui_m['event_count'], alpha=0.07, color=COLORS['EUI'])

ax1.set_title('Core Application Usage — Monthly Events (RUI vs EUI vs CDE)', fontsize=13, pad=10)
ax1.set_ylabel('Event Count')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.legend(loc='upper left', fontsize=11)

# ── Bottom: Unique users — the growth story ──────────────────────────────────
ax2 = axes[1]
ax2.plot(rui_m['event_month'], rui_m['unique_users'],
         color=COLORS['RUI'], linewidth=2.5, marker='o', markersize=3.5, label='RUI')
ax2.plot(eui_m['event_month'], eui_m['unique_users'],
         color=COLORS['EUI'], linewidth=2.5, marker='s', markersize=3.5, label='EUI')
ax2.plot(cde_m['event_month'], cde_m['unique_users'],
         color=COLORS['CDE'], linewidth=2, marker='^', markersize=3.5, label='CDE', linestyle='--')

ax2.fill_between(eui_m['event_month'], eui_m['unique_users'], alpha=0.1, color=COLORS['EUI'])

# Annotate EUI overtaking RUI in unique users
eui_dec25 = eui_m[eui_m['event_month']=='2025-12-01']['unique_users'].values
rui_dec25 = rui_m[rui_m['event_month']=='2025-12-01']['unique_users'].values
if len(eui_dec25) and len(rui_dec25):
    if eui_dec25[0] > rui_dec25[0]:
        ax2.annotate('EUI surpasses RUI\nin unique users (Dec 2025)',
                     xy=(pd.Timestamp('2025-12-01'), eui_dec25[0]),
                     xytext=(pd.Timestamp('2025-04-01'), eui_dec25[0] + 100),
                     arrowprops=dict(arrowstyle='->', color='#555', lw=1.3),
                     fontsize=9, color='#555',
                     bbox=dict(boxstyle='round,pad=0.3', fc='#fff3cd', ec='#ffaa00', alpha=0.85))

ax2.set_title('Unique Users per Month — EUI User Base Growing Fastest', fontsize=13, pad=10)
ax2.set_ylabel('Unique Users')
ax2.set_xlabel('Month')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.tick_params(axis='x', rotation=40)
ax2.legend(loc='upper left', fontsize=11)

fig.suptitle('Fig 7. RUI vs EUI vs CDE — Full Platform Comparison (Oct 2023 – Jan 2026)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig7_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** The combined view reveals the most important narrative of the HRA Portal:
1. **RUI dominates event volume** — it generates bursty, high-intensity usage (tissue registration sessions are heavy API consumers), but unique users per month are plateauing around 1,000–1,400.
2. **EUI is quietly winning the user growth race** — while events are relatively stable, unique users grew from ~450 to ~2,900 over the study period. By late 2025, EUI is rivalling RUI in unique users despite having far fewer events per user session.
3. **CDE remains critically underutilized** — averaging under 600 events per month throughout the whole period, with only ~200–350 unique users.

### 4.2 Year-over-Year Change Summary

In [ ]:
# ── YoY Summary Table ────────────────────────────────────────────────────────
def yoy(df, year_a, year_b, col):
    a = df[df.event_month.dt.year == year_a][col].sum()
    b = df[df.event_month.dt.year == year_b][col].sum()
    return a, b, (b-a)/a*100 if a else 0

summary_rows = []
for iface, df in [('RUI', rui_m), ('EUI', eui_m), ('CDE', cde_m)]:
    e24, e25, ep = yoy(df, 2024, 2025, 'event_count')
    u24, u25, up = yoy(df, 2024, 2025, 'unique_users')
    summary_rows.append({'Interface': iface,
                         'Events 2024': f'{e24:,}', 'Events 2025': f'{e25:,}',
                         'Event Δ': f'{ep:+.1f}%',
                         'Users 2024': f'{u24:,}', 'Users 2025': f'{u25:,}',
                         'User Δ': f'{up:+.1f}%'})

summary_df = pd.DataFrame(summary_rows)
print('=== Year-over-Year Summary (2024 vs 2025) ===')
print(summary_df.to_string(index=False))

# ── Bar chart of YoY event change ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

interfaces = ['RUI', 'EUI', 'CDE']
event_2024 = [391150, 200752, 5093]
event_2025 = [377493, 210008, 5394]
user_2024  = [rui_m[rui_m.event_month.dt.year==2024]['unique_users'].sum(),
              eui_m[eui_m.event_month.dt.year==2024]['unique_users'].sum(),
              cde_m[cde_m.event_month.dt.year==2024]['unique_users'].sum()]
user_2025  = [rui_m[rui_m.event_month.dt.year==2025]['unique_users'].sum(),
              eui_m[eui_m.event_month.dt.year==2025]['unique_users'].sum(),
              cde_m[cde_m.event_month.dt.year==2025]['unique_users'].sum()]

x = np.arange(3)
w = 0.35

# Events
ax1 = axes[0]
ax1.bar(x - w/2, event_2024, w, label='2024', color='#9ecae1', edgecolor='white')
ax1.bar(x + w/2, event_2025, w, label='2025', color='#2171b5', edgecolor='white')
ax1.set_xticks(x); ax1.set_xticklabels(interfaces, fontsize=12)
ax1.set_ylabel('Total Events'); ax1.set_title('Total Events: 2024 vs 2025')
ax1.legend()
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))

# Users
ax2 = axes[1]
ax2.bar(x - w/2, user_2024, w, label='2024', color='#fdae6b', edgecolor='white')
ax2.bar(x + w/2, user_2025, w, label='2025', color='#d94801', edgecolor='white')
ax2.set_xticks(x); ax2.set_xticklabels(interfaces, fontsize=12)
ax2.set_ylabel('Total Unique Users'); ax2.set_title('Unique Users: 2024 vs 2025')
ax2.legend()
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))

fig.suptitle('Fig 8. Year-over-Year Performance — 2024 vs 2025 per Interface', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_yoy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Summary of Key Findings

### RUI Findings
1. **Total events (Oct 2023–Jan 2026): 818,780** — the most event-intensive interface.
2. **Peak usage: November 2024 (~189K events)** — strong signal of a coordinated HuBMAP event or data release cycle.
3. **Tissue Collision Check accounts for 53% of RUI traffic** — reflects real-time iterative placement behavior, not passive browsing.
4. **Spatial Placement calls: ~93K (11.3% of RUI events)** — consistently used; funnel drop-off from collision to placement is ~4.7:1 (healthy for an iterative UX workflow).
5. **Organ usage is well-distributed** — Large Intestine, Spleen, and Kidney lead, but all 8 organs have meaningful activity.
6. **Unique users growing**: ~300/month (2023) → ~1,100–1,400/month (late 2025).

### EUI Findings  
1. **Total events (Oct 2023–Jan 2026): 607,043** — steady, consistent platform.
2. **User growth is the key EUI story**: unique users grew ~200% from 2024 to 2025.
3. **Cell Type and Anatomical Structure filters are the most-used EUI features** (40K and 29K events respectively) — users engage with the scientific data layer, not just the 3D visualization.
4. **All tree-navigation UI components** (cell-type-tree, ontology-tree, biomarker-tree) are actively used, confirming hierarchical exploration is a core EUI workflow.
5. **EUI surpasses RUI in unique users by late 2025** — a major crossover moment suggesting EUI is becoming the wider-reach interface.

### Comparative Findings
- **RUI = depth**, **EUI = breadth**: RUI has fewer, more intensive sessions; EUI has more users with lighter sessions.
- **CDE remains critically underutilized** at <1% of core platform events — unchanged across the full observation period.
- **Platform is growing**: total unique users across all core interfaces increased significantly from 2024 to 2025.

---
## 6. Recommendations for the HRA Development Team

Based on the RUI + EUI analysis:

| Finding | Recommendation |
|---|---|
| Nov 2024 RUI spike likely event-driven | Build event-aware analytics; tag known release dates in the dataset for richer correlation |
| Spatial Placement funnel drop-off ~4.7:1 | Investigate whether users who abandon after collision-check stage encounter UI friction; consider adding a guided completion prompt |
| Large Intestine & Spleen lead organ registrations | Prioritize these organ models for quality updates and new version releases |
| EUI user base growing 200% YoY | Invest in onboarding flows to convert new EUI visitors into returning power users |
| Tree UI components are all actively used | Ensure these are well-maintained and not deprecated in future UI refactors |
| CDE barely registers | CDE needs either: a dedicated onboarding campaign, surfacing within EUI/RUI workflows, or a UX audit |
